# Parse new statements & rebuild summary

Pipeline (see `README.md`)
Run the cells top to bottom.

In [ ]:
# Setup: auto-reload the project modules so edits take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

from stmt import parseStatement, reconcileUpdate, regroupUpdate, OUT_DIR
from trex import summarize, checkSummaryTotal

YEAR = 2026  # statements / summary year
SUMMARY = f'summary{YEAR}.csv'

## 1. Parse the new statements

Each call extracts the PDF, categorizes the transactions, and writes `statements_parsed/<name>.csv`.
Watch the printout for `AUTO-ADDED` (a new category rule was guessed) and any `[skip credit]` lines.

In [ ]:
NEW = ['Chase05', 'PLG05', 'PLY0405', 'UOB05']

for name in NEW:
    print(f'\n===== {name} =====')
    parseStatement(name)

## 2. (Optional) Review & fix categories

Open a `statements_parsed/<name>.csv` and look for the **uncategorized** section or wrong categories.
To correct: copy the file to `statements_parsed/<name>-update.csv`, edit the *category* column on the rows you want
to change (keep the grand total the same), then reconcile.

`reconcileUpdate` teaches `categories/cat.csv` the new mapping, rewrites `<name>.csv`, and removes the `-update.csv`.

In [ ]:
# Show any pending *-update.csv files waiting to be reconciled.
import glob, os
pending = sorted(os.path.basename(p)[:-len('-update.csv')]
                 for p in glob.glob(f'{OUT_DIR}/*-update.csv'))
print('pending -update files:', pending or '(none)')

In [ ]:
# Reconcile a specific statement's edits. Set the name, then run.
# (Skip this cell if you made no manual edits.)

reconcileUpdate('UOB05')

## 3. Rebuild the summary

Consolidate every `statements_parsed/*.csv` into the category x month table and verify the grand total
matches the sum of the parsed rows (CHASE converted USD->SGD).

In [ ]:
summarize(year=YEAR)
print()
checkSummaryTotal(summary_name=SUMMARY, year=YEAR)